In [1]:
import numpy as np
import pandas as pd
import os
import string
import re

pd.set_option('display.max_columns', None)

In [2]:
!pip install tqdm

In [2]:
from tqdm import tqdm

In [4]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/pickle-part-three/pixel_part_three
/kaggle/input/dom-data/dom_pickle
/kaggle/input/wsdm-cup-multilingual-chatbot-arena/sample_submission.csv
/kaggle/input/wsdm-cup-multilingual-chatbot-arena/train.parquet
/kaggle/input/wsdm-cup-multilingual-chatbot-arena/test.parquet
/kaggle/input/pickle-part-two/pixel_part_two


In [15]:
# df_sample = pd.read_csv("/kaggle/input/wsdm-cup-multilingual-chatbot-arena/sample_submission.csv")
df_train =pd.read_parquet("/kaggle/input/wsdm-cup-multilingual-chatbot-arena/train.parquet")
df_test = pd.read_parquet("/kaggle/input/wsdm-cup-multilingual-chatbot-arena/test.parquet")
df=df_train

### Code Detection 

In [4]:
def contains_code(text):
    code_patterns = [
        r'```[\s\S]+?```',          
        r'`[^`]+`',          
        r'(?m)^(?: {4}|\t).+',
        r'\bdef\b|\bclass\b|\bimport\b|\breturn\b|\bprint\b|\bfor\b|\bwhile\b|\bif\b|\belse\b',
        r'<[a-zA-Z/][^>]*?>',          
    ]

    text = text.strip()

    for pattern in code_patterns:
        if re.search(pattern, text):
            if re.search(r'`[^`]+`', text):
                if not re.search(r'`[a-zA-Z0-9\s]+`', text):
                    return True
            else:
                return True
    return False

In [5]:
def first_text_categorization(row):
    prompt = row['prompt']
    response_a = row['response_a']
    response_b = row['response_b']
    
    prompt_bool = contains_code(prompt.lower())
    response_a_bool = contains_code(response_a.lower())
    response_b_bool = contains_code(response_b.lower())
    
    text_bool = prompt_bool or response_a_bool or response_b_bool
    return prompt_bool,response_a_bool,response_b_bool,text_bool

In [9]:
tqdm.pandas()
cs_data = list(df.progress_apply(first_text_categorization, axis=1))

100%|██████████| 48439/48439 [00:09<00:00, 5327.37it/s]


In [10]:
df['cs_data'] = cs_data
df['code_in_prompt']= df['cs_data'].apply(lambda x: x[0])
df['code_in_response_a']= df['cs_data'].apply(lambda x: x[1])
df['code_in_response_b']= df['cs_data'].apply(lambda x: x[2])
df['code']= df['cs_data'].apply(lambda x: x[3])
del(df['cs_data'])

In [12]:
def contains_code_xtreme(text):
    code_patterns = [
        r'[{}();=\[\]:<>]',
        r'#include\s*<.+>',        
        r'\bdef\s+\w+\s*\(.*\):',
        r'\bclass\s+\w+\s*\(?.*?\)?:',        
        r'```[a-zA-Z]*\n.*?```',
        r'^[ \t]*\w+.*\(.*\).*{',
    ]

    matches = sum(bool(re.search(pattern, text, re.MULTILINE)) for pattern in code_patterns)
    
    if matches >= 2:
        return True

    if re.search(r'```.*?```|`[^`]+`', text, re.MULTILINE):
        return True

    lines = text.split('\n')
    symbol_count_threshold = 5
    for line in lines:
        symbols = re.findall(r'[{}();=\[\]:<>]', line)
        if len(symbols) >= symbol_count_threshold:
            return True

    return False

In [13]:
def main_contains_code_xtreme(row):
    prompt = row['prompt']
    response_a = row['response_a']
    response_b = row['response_b']

    prompt_bool = row['code_in_prompt']
    response_a_bool = row['code_in_response_a']
    response_b_bool = row['code_in_response_b']
    text_bool = row['code']
    
    if text_bool==False:
        return False,False,False,False

    if prompt_bool:
        prompt_bool = contains_code_xtreme(prompt)
    if response_a_bool:
        response_a_bool = contains_code_xtreme(response_a)
    if response_b_bool:
        response_b_bool = contains_code_xtreme(response_b)
    
    text_bool = prompt_bool or response_a_bool or response_b_bool
    return prompt_bool,response_a_bool,response_b_bool,text_bool

In [14]:
len(df[df['code']==True]) , len(df[df['code_in_prompt']==True]) , len(df[df['code_in_response_a']==True]) , len(df[df['code_in_response_b']==True])

(28353, 12985, 20609, 20651)

In [15]:
tqdm.pandas()
x_cs_data = list(df.progress_apply(main_contains_code_xtreme, axis=1))

100%|██████████| 48439/48439 [00:14<00:00, 3421.53it/s]


In [20]:
df['x_cs_data'] = x_cs_data
df['code_in_prompt']= df['x_cs_data'].apply(lambda x: x[0])
df['code_in_response_a']= df['x_cs_data'].apply(lambda x: x[1])
df['code_in_response_b']= df['x_cs_data'].apply(lambda x: x[2])
df['code']= df['x_cs_data'].apply(lambda x: x[3])
del(df['x_cs_data'])

In [21]:
len(df[df['code']==True]) , len(df[df['code_in_prompt']==True]) , len(df[df['code_in_response_a']==True]) , len(df[df['code_in_response_b']==True])

(12082, 3993, 6154, 6057)

### Code Language detection for Code = True

In [44]:
import re

def detect_code_language_details(text):

    web_dev_pattern = re.compile(
    r'(?i)(?:'
    r'\{[^{}]*?\b[a-zA-Z-]+\s*:\s*[^;{}]+;\s*\}|'  
    r'<style\b[^>]*>[^<]*?\{[^{}]*?\b[a-zA-Z-]+\s*:\s*[^;{}]+;\s*\}[^<]*<\/style>|'
    r'\b(function|var|let|const|=>|return|if|else|for|while|class)\b|'  
    r'<[a-zA-Z!\/][^>]*>'
    r')'
) 
    python_pattern = re.compile(r"\b(def|class|import|from|try|except|with|open|input|print|len|map|filter|zip|lambda|enumerate|range|dict|list|set|tuple|pandas|numpy|matplotlib|seaborn|scikit-learn|tensorflow|keras|torch|DataFrame|Series|read_csv|read_excel|head|tail|describe|groupby|merge|concat|plot|scatter|hist|bar|boxplot|mean|median|std|min|max|sum|apply|transform|pivot_table|correlation|train_test_split|fit|predict|accuracy_score|confusion_matrix|classification_report|roc_curve|auc)\b")
    java_pattern = re.compile(r"\b(public|private|protected|class|void|String|new|try|catch)\b")
    cpp_pattern = re.compile(r"\b(int|float|double|char|class|#include)\b")
    comprehensive_pattern = re.compile(r"\b(lambda|print|System\.out\.println|cout|cin|alert)\b")

    web_dev_matches = web_dev_pattern.findall(text)
    python_matches = python_pattern.findall(text)
    java_matches = java_pattern.findall(text)
    cpp_matches = cpp_pattern.findall(text)
    comprehensive_matches = comprehensive_pattern.findall(text)

    filtered_python_matches = [m for m in python_matches if m not in {"if", "return", "for", "while"}]
    filtered_java_matches = [m for m in java_matches if m not in {"if", "return", "for", "while"}]
    filtered_cpp_matches = [m for m in cpp_matches if m not in {"if", "return", "for", "while"}]

    lang_list = []
    if len(list(set(web_dev_matches)))>0:
        lang_list.append(1)
    else:
        lang_list.append(0)
        
    if len(list(set(python_matches)))>0 or len(list(set(java_matches)))>0 or len(list(set(cpp_matches)))>0:
        lang_list.append(1)
    else:
        lang_list.append(0)
        
    if len(list(set(comprehensive_matches)))>0:
        lang_list.append(1)
    else:
        lang_list.append(0)

    return lang_list

In [45]:
# "Web-dev", ["Python" | "Java" | "C++"], "Comprehensive"

def detect_code_language(row):
    if row['code']==True:
        code_languages_in_prompt = detect_code_language_details(row['prompt'])
        code_languages_in_response_a =  detect_code_language_details(row['response_a'])
        code_languages_in_response_b =  detect_code_language_details(row['response_b'])
        return [code_languages_in_prompt,code_languages_in_response_a,code_languages_in_response_b]
    else:
        code_languages_in_prompt=[0,0,0]
        code_languages_in_response_a=[0,0,0]
        code_languages_in_response_b=[0,0,0]
        return [code_languages_in_prompt,code_languages_in_response_a,code_languages_in_response_b]

In [46]:
df[["code_languages_in_prompt", "code_languages_in_response_a", "code_languages_in_response_b"]] = df.apply(detect_code_language, axis=1, result_type="expand")

### Light Preprocessing for prompt - removing punctuations

In [26]:
def clean_for_segmentation(row):    
    text = row['prompt']
    characters_to_remove = r"[-.,\n'\":!?]"
    cleaned_text = re.sub(characters_to_remove, "", text)
    final_text = re.sub(r"\s+", " ", cleaned_text).strip()
    return final_text

In [27]:
df['ft_prompt'] = df.apply(clean_for_segmentation,axis=1)

### Segmentation

In [28]:
df["prompt_len"] = df['ft_prompt'].apply(lambda x: len(x.split(' ')))

In [29]:
def segmentation(row):
    sentences=[]
    prompt_len = row['prompt_len']
    prompt = row['ft_prompt']
    prompt_arr = prompt.split(' ')
    segment = 6
    if prompt_len<6:
        text = " ".join(prompt_arr[0:prompt_len])
        sentences.append(text)
    else:
        for i in range(0,prompt_len,segment):
            if i+(2*segment)>prompt_len:
                text = " ".join(prompt_arr[i:prompt_len])
                sentences.append(text)
                break
            else:    
                text = " ".join(prompt_arr[i:i+segment])
                sentences.append(text)
    return sentences

In [30]:
df['prompt_segements'] = df.apply(segmentation,axis=1)

### Dominating language

In [31]:
!pip install langcodes
!pip install langdetect 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.9 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993221 sha256=b48a0aca99302a42f043ff5446c672f0af2d82b55b1de6ef144bcc12b3dfe8ba
  Stored in directory: /root/.cache/pip/wheels/95/03/7d/59ea870c70ce4e5a370638b5462a7711ab78fba2f655d05106
Successfully built langdetect


In [32]:
from nltk.corpus import stopwords
import nltk 

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [33]:
import langcodes
from langdetect import detect, detect_langs

In [34]:
def lang_detection_prompt(row):
    sentences = row['prompt_segements']
    extra_stopwords = []
    language_dict = {}
    language_frequency = {}
    language_weight = {}
    total= 0
    filtered_prompt = ''
    
    for sentence in sentences:
        try:
            detected_code = detect(sentence) 
            detected_language = langcodes.Language.get(detected_code).display_name()

            if detected_language in language_frequency:
                language_frequency[detected_language] += 1
            else:
                language_frequency[detected_language] = 1

            if (detected_language is None) or (detected_language ==''):
                detected_language = 'english'
            
        except Exception as e:
            break
    
    for key in list(language_frequency.keys()):
        total = total+ language_frequency[key]

    for key in list(language_frequency.keys()):
        language_weight[key]= language_frequency[key]/total

    # return [filtered_prompt,language_frequency,language_weight]
    return [language_frequency,language_weight]

In [35]:
tqdm.pandas()
ls_prompt = list(df.progress_apply(lang_detection_prompt, axis=1))

100%|██████████| 48439/48439 [1:37:03<00:00,  8.32it/s]  


In [36]:
df['prompt_meta'] = ls_prompt
df['prompt_counter']= df['prompt_meta'].apply(lambda x: x[0])
df['prompt_weight']= df['prompt_meta'].apply(lambda x: x[1])
del(df['prompt_meta'])

In [37]:
df.head()

,id,prompt,response_a,response_b,winner,model_a,model_b,language,code_in_prompt,code_in_response_a,code_in_response_b,code,code_languages_in_prompt,code_languages_in_response_a,code_languages_in_response_b,ft_prompt,prompt_len,prompt_segements,prompt_counter,prompt_weight
0,00007cff95d7f7974642a785aca248b0f26e60d3312fac...,vieš po Slovensky?,"Áno, hovorím po slovensky. Ako vám môžem pomôcť?","Áno, veď som tu! Môžem ti pomôcť s otázkami al...",model_a,o1-preview,reka-core-20240904,Slovak,False,False,False,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]",vieš po Slovensky,3,[vieš po Slovensky],{'Slovak': 1},{'Slovak': 1.0}
1,00010ed04b536f56ebe43eef1100c13906abea12bf9855...,You will be given a piece of news. Analyze it ...,Let's break down the news and analyze it accor...,"```json\n{\n ""contains_orgs"": true,\n ""orgs""...",model_a,gemma-2-27b-it,gemini-1.5-flash-002,Russian,False,True,True,True,"[1, 1, 0, 0, 0]","[1, 1, 0, 0, 0]","[0, 0, 0, 0, 0]",You will be given a piece of news Analyze it a...,357,"[You will be given a piece, of news Analyze it...","{'English': 20, 'Russian': 39}","{'English': 0.3389830508474576, 'Russian': 0.6..."
2,0003800d510e38803efba5ceaec122bc66408fe367b0be...,"Dört basamaklı, rakamları birbirinden ve sıfır...","Bu soruyu çözmek için, verilen koşulları adım ...",Bu problemi adım adım çözelim:\n\n1) ABCD - DC...,model_a,gpt-4-0125-preview,claude-3-5-sonnet-20240620,Turkish,False,False,False,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]",Dört basamaklı rakamları birbirinden ve sıfırd...,31,[Dört basamaklı rakamları birbirinden ve sıfır...,{'Turkish': 5},{'Turkish': 1.0}
3,00072026c68f5418ef2da238394e418ce72a534b9b22d5...,"현재 추천된 탑 3 종목인 Cabaletta Bio (CABA), Rocket Ph...","죄송하지만 저는 금융 조언을 제공할 수 없습니다. 저는 AI 모델이며, 투자 결정에...",현재 추천된 탑 3 종목에 순위를 매기기 위해서는 여러 가지 요소들을 고려해야 합니...,model_b,gemma-2-2b-it,llama-3.1-nemotron-70b-instruct,English,False,False,False,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]",현재 추천된 탑 3 종목인 Cabaletta Bio (CABA) Rocket Pha...,15,"[현재 추천된 탑 3 종목인 Cabaletta, Bio (CABA) Rocket P...","{'Korean': 1, 'English': 1}","{'Korean': 0.5, 'English': 0.5}"
4,0007ce7cf6bc1b5a8f8a4669b854fb12030863c970d9dc...,Please be boring,"Alright, I'll be as boring as possible.\n\nTod...","Understood. Here is a straightforward, unadorn...",model_a,reka-flash-20240722,grok-2-2024-08-13,English,False,False,False,False,"[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]","[0, 0, 0, 0, 0]",Please be boring,3,[Please be boring],{'English': 1},{'English': 1.0}


In [11]:
# load pickle file here ------X---------

In [21]:
# import pickle

# path = '/kaggle/input/dom-data/dom_pickle'

# try:
#     with open(path, 'rb') as f:
#         df = pickle.load(f)
#     print("Pickle file loaded successfully!")
# except Exception as e:
#     print(f"Error loading pickle file: {e}")

Pickle file loaded successfully!


In [38]:
# later : just check when dictionary have languages with equal ratio then care 

def get_dom_language(row):
    counter = row['prompt_counter']
    if counter:
        max_pair = max(counter.items(), key=lambda x: (x[1], x[0] == "english"))
        return max_pair[0]
    else:
        print(counter,' : ',row['id'],' : ',row.index[0])

In [39]:
tqdm.pandas()
df['dom_lang'] = df.progress_apply(get_dom_language, axis=1)

 30%|██▉       | 14349/48439 [00:00<00:00, 77432.06it/s]

{}  :  01614dda88361a6c798c8e7c98f3bcaef620952b2f5e937712f5a5bcd2a0e673  :  id
{}  :  03e57995fca216a372b3d8722fc0726f182f6ccf15635b5cea9e3aad2c2658ce  :  id
{}  :  0c771722678945793a14015e5f562d8c027c3bb0a844a1fba9852e66c96b2d67  :  id
{}  :  0cb443e2567ad409963c23da63a7187525460bfc9992f089337a3fed29f3d4a9  :  id
{}  :  0ee2f2b81d501a253bc2bb5ceb95d976714a86511732987c3ca82e5dbe614dbe  :  id
{}  :  100c380f1b5487e8ada5960a00e5ba37c259482b542e15e5d3e13aabe08bb8b5  :  id
{}  :  10dbae7b441799bdffdc44580cf20fa6a56b59aef5e68c00f0b70d88e1bbf6a5  :  id
{}  :  1126b35fc8bd99fc6bd7dca3d544de2f36a24a4b4f143f2e2c3da6b2a7d9e637  :  id
{}  :  12aa7402d588ed91a6a7ee273de258501413c464e6e3176db8776bebce7ded11  :  id
{}  :  1339b68bb10f0c530673284c42654c22154a4200deecb58ab857bf0c3d08a847  :  id
{}  :  145ba6104ed58c0a7b35d95cdf8ad3b26ab7ec7ea5aa088623cbdc91316d710e  :  id
{}  :  1aea39cd4b4d24a73646739bd08359d172bac3c92cd3c45bc6fe9653c8a67535  :  id
{}  :  1b3ec9f4b9b505c106c02e00b90fa086b63c83f0f6bf9

 71%|███████   | 34155/48439 [00:00<00:00, 90842.24it/s]

{}  :  73e21ed1dd488e615af9f3b3ecc15472c4fb8da8cff95240255151c4dbe3ff99  :  id
{}  :  77994390e5df247379fad986167145bd17fc0f5f1db9d5a5c5a8dec4dc5eea3b  :  id
{}  :  7a2b542c518df0d95ffe17de3b7d966adf960822536e073739ca1c417f42f389  :  id
{}  :  7c397cdd73360e486e475baeab5a30a33518e3af377a3123c8261152c1751177  :  id
{}  :  7dcf37938910d09a0cbbcc5a12affad0ff9f4e6e187774a6ae801b1002ceb7e7  :  id
{}  :  7e2e154e968f8538b578808c3de0e79e642881b9308381f47bb73cb276a6d4df  :  id
{}  :  7fc46f98bb4036cabd85e67a4f829a4cd040cf4d49c544399cfad96891fec0cb  :  id
{}  :  8200d5970ab16b2632d275788d3ac403fbfe1516be22da1c3c255ea5c7f03ded  :  id
{}  :  85ffa8305f6bb1b1a5d003b5197422232fd73a24b3dfa7b3828388dabbc1e2c6  :  id
{}  :  87e73322a3cfda4493bd7f98f883437290897fdf2000c4a77428bdde18601f6a  :  id
{}  :  8db964b5e124535e9ba1a2aaf74925b07aadc893fa5588ff5933b044e78fbd3e  :  id
{}  :  90b7c8d2af91eaabd77f93709f4c4eb07bcf5fe4886e351a92420a67544cf4a8  :  id
{}  :  92364c8e4f687a396e7d14e916e8ce64a4946ea4abb73

100%|██████████| 48439/48439 [00:00<00:00, 84142.19it/s]

id
{}  :  d928815b7d244172edb91a65d0d1aa6cdd2cb0ca58b9f2c3b97b8457d1549b06  :  id
{}  :  d9bfcc7155822310eeeb5a46d4a6452a8feb1a6be1f23ed3622f476b8d9e1483  :  id
{}  :  da124480bf4c279249a6b03d3877b9698faa802d47994fd2f29753ec937f9cb4  :  id
{}  :  db76d177d27b4769575d459ca4771c318f49f05f69075e6dbe1a03dd432686a6  :  id
{}  :  dce48443d3a1522b6b71c47cbeeff17f6e09d2d714619139d371e3d8203ff758  :  id
{}  :  ded27fe1ab6f2b464ebf8fe819e6b2a9efec491d775ea316172ac7fdf276f175  :  id
{}  :  e50b16f992786e5b4e550808d717b999ab3da072397e8ad9d5ed7efa83d92ffb  :  id
{}  :  eeddfd3efa9857b9f537cd41999e09a92cba9d54dd5bd0e3f5e43046e9aef4b5  :  id
{}  :  efb264cff8b8d35d300dfa4847e1d412d0f1308a8a721f300492a199829d65ee  :  id
{}  :  f006ae6c3f8e53ec38f42bd517ad489593fadb39dc4dfe92fc85de772d4291aa  :  id
{}  :  f08fb69d49a38f6e81a95a7f38bf5ff7b14fb2c455e989ac8d84624f41e1e52b  :  id
{}  :  f1edaafbc557db0c4b01dbc74e88844a28434725ce2e193754e3f2821e3a25c2  :  id
{}  :  f4fe626c3e1ddc5d9ba1d65944a7f8ae8f9ecb9dac

### Final categorization

In [46]:
def contains_mathematical_operations(text, threshold=0.3):
    numbers = re.findall(r'\d+(\.\d+)?', text)
    operators = re.findall(r'[+\-*/=]', text)
    equation_pattern = r'(\d+=)+\d+'
    equations = re.findall(equation_pattern, text)
    math_expression_pattern = r'(\d+[+\-*/=]\d+)'
    math_expressions = re.findall(math_expression_pattern, text)
    total_chars = len(text)
    try:
        num_chars = sum(len(num) for num in numbers)
        if total_chars<=0:
            return False
        proportion_of_numbers = num_chars / total_chars
        if proportion_of_numbers > threshold or len(equations) > 0 or len(math_expressions) > 0:
            return True
        else:
            return False
    except:
        print(text)

In [47]:
def text_code_unknown_categorization(row):
    if row['code']:
        return 'code'
    if contains_mathematical_operations(row['ft_prompt']) or contains_mathematical_operations(row['response_a']) or contains_mathematical_operations(row['response_b']):
        return 'logical_operation'
    counter = row['prompt_counter']
    if counter:
        return 'text'
    else:
        return 'unknown'

In [48]:
tqdm.pandas()
df['type'] = df.progress_apply(text_code_unknown_categorization, axis=1)

100%|██████████| 48439/48439 [00:18<00:00, 2603.02it/s]


### Translation to dominating language

In [53]:
!pip install googletrans==4.0.0-rc1

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 29.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 3.1 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17397 sha256=3f8123a055a25c4700ba1528f427bbe4fec81eb2eb8a84d308ccaf4e80d4e066
  Stored in directory: /root/.cache/pip/wheels/c0/59/9f/7372f0cf70160fe61b528532e1a7c8498c4becd6bcffb022de
Successfully built googletrans
  Attempting uninstall: chardet
    Found existing installation: chardet 5.2.0
    Uninstalling chardet-5.

In [54]:
from googletrans import Translator, LANGUAGES

translator = Translator()

In [56]:
def translation(row):
        
    source_language = "auto"
    target_language = row['dom_lang']  
    if target_language=='None' or target_language==None:
        target_language='english'
        
    response_a = row['response_a']
    response_b = row['response_b']
    prompt = row['ft_prompt']
    
    if row['type']=='unknown' or row['type']=='code' or row['type']=='logical_operation':
        return prompt,response_a,response_b
  
    src_code = next((code for code, name in LANGUAGES.items() if name.lower() == source_language.lower()), None)
    dest_code = next((code for code, name in LANGUAGES.items() if name.lower() == target_language.lower()), None)
    
    if src_code and dest_code:
        tr_prompt = translator.translate(prompt, src=src_code, dest=dest_code)
        tr_response_a = translator.translate(response_a, src=src_code, dest=dest_code)
        tr_response_b= translator.translate(response_b, src=src_code, dest=dest_code)
        return tr_prompt.text,tr_response_a.text,tr_response_b.text
    else:
        return prompt,response_a,response_b

In [57]:
tqdm.pandas()
tr_list = list(df.progress_apply(translation, axis=1))

100%|██████████| 48439/48439 [00:02<00:00, 21807.89it/s]


In [58]:
df['tr_meta'] = tr_list
df['tr_prompt']= df['tr_meta'].apply(lambda x: x[0])
df['tr_response_a']= df['tr_meta'].apply(lambda x: x[1])
df['tr_response_b']= df['tr_meta'].apply(lambda x: x[2])
del(df['tr_meta'])

### Weighted sum calculation for both the responses

In [11]:
!pip install polyglot
!pip install PyICU
!pip install pycld2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.3/126.3 kB 6.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for polyglot: filename=polyglot-16.7.4-py2.py3-none-any.whl size=52561 sha256=ac2cb26efa9efe6149a2f46c10def1fbcc8efa1d539177136d4c829269fb0b58
  Stored in directory: /root/.cache/pip/wheels/aa/92/4a/b172589446ba537db3bdb9a1f2204f27fe71217981c14ac368
Successfully built polyglot
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 263.9/263.9 kB 12.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for PyICU: filename=PyICU-2.14-cp310-cp310-linux_x86_64.whl size=1809840 sha256=2374ccdb19f1191f10b56e510311aee940403f33262c7478e5d5c0d4289f608c
  Stored in directory: /root/.cache/pip/wheels/78/6e/76/17c73021179c06c29d9b108896b9248da0de4f2af93f63d405
Successfully built PyICU
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 MB 4.9 MB/s eta 

In [15]:
def clean_before_wt_sum(text):    
    characters_to_remove = r"[-.,\n'\":!?]"
    cleaned_text = re.sub(characters_to_remove, "", text)
    final_text = re.sub(r"\s+", " ", cleaned_text).strip()
    return final_text

In [66]:
from polyglot.detect import Detector
from charset_normalizer import from_bytes

def club_detection(row):
    
    if row['type']=='code' or row['type']=='unknown' or row['type']=='logical_operation':
        return -1,-1,{},{}
    
    response_a_language_frequency={}
    response_b_language_frequency={}
    response_a_weighted_sum=0
    response_b_weighted_sum=0
      
    set_weight = row['prompt_weight']
    response_a = clean_before_wt_sum(row['response_a'])
    response_b = clean_before_wt_sum(row['response_b'])
    
    try:
        response_a_detector = Detector(response_a, quiet=True)
        response_b_detector = Detector(response_b, quiet=True)
        
        for language in response_a_detector.languages:
            if language.read_bytes==0 or language.name=='un':
                proportion=0
            else:
                proportion = language.read_bytes / sum(lang.read_bytes for lang in response_a_detector.languages)
            if (language.name not in response_a_language_frequency) and language.name!='un':
                response_a_language_frequency[language.name]=proportion
    
        for language in response_b_detector.languages:
            if language.read_bytes==0 or language.name=='un':
                proportion=0
            else:
                proportion = language.read_bytes / sum(lang.read_bytes for lang in response_b_detector.languages)
            if (language.name not in response_b_language_frequency) and language.name!='un':
                response_b_language_frequency[language.name]=proportion
    
        for key in list(response_a_language_frequency.keys()):
            if key in list(set_weight.keys()):
                response_a_weighted_sum = response_a_weighted_sum + set_weight[key]*response_a_language_frequency[key]
            else:
                response_a_weighted_sum = response_a_weighted_sum + 0*response_a_language_frequency[key]
    
        for key in list(response_b_language_frequency.keys()):
            if key in list(set_weight.keys()):
                response_b_weighted_sum = response_b_weighted_sum + set_weight[key]*response_b_language_frequency[key]
            else:
                response_b_weighted_sum = response_b_weighted_sum + 0*response_b_language_frequency[key]

    except:
        print(row['id'])
        # print(response_b_detector)

    return response_a_weighted_sum,response_b_weighted_sum,response_a_language_frequency,response_b_language_frequency

In [67]:
tqdm.pandas()
wt_list = list(df.progress_apply(club_detection, axis=1))

  9%|▉         | 4581/48439 [00:02<00:20, 2134.47it/s]

16495abe725138b61e2461a3e4cc35a0ed4eaaf393f1e2b1795f78e988d5fdb7


 14%|█▍        | 6705/48439 [00:03<00:18, 2286.19it/s]

2122e0b589c878a9fad3c522ec911f4570615fd10f28ede535399f062dd63f8c


 22%|██▏       | 10661/48439 [00:04<00:16, 2266.87it/s]

...

b1a2b39bdfc22cf135f7b686c12c3e4bc86eb7b80ceed4936a692377c7d857e4


 88%|████████▊ | 42485/48439 [00:19<00:02, 2130.91it/s]

df0d6ea8667bbaa31daab6a4b1990445cdd83f9eb8cff90356c8eeb013062f26


100%|██████████| 48439/48439 [00:22<00:00, 2184.87it/s]

fecd4f4c62c0d4c47b7eca485a1e3b2811d59cef3d2957bee94a2399dabc4d5a


In [68]:
df['wt_meta'] = wt_list
df['wt_sum_response_a']= df['wt_meta'].apply(lambda x: x[0])
df['wt_sum_response_b']= df['wt_meta'].apply(lambda x: x[1])
df['counter_response_a']= df['wt_meta'].apply(lambda x: x[2])
df['counter_response_b']= df['wt_meta'].apply(lambda x: x[3])
del(df['wt_meta'])

In [71]:
len(df[df['type']=='text']) , len(df[df['type']=='code']) , len(df[df['type']=='logical_operation']) , len(df[df['type']=='unknown'])

(32092, 12082, 4214, 51)

In [12]:
def club_detection_for_code(row):
    
    if row['type']=='text' or row['type']=='unknown' or row['type']=='logical_operation':
        return 	row['wt_sum_response_a'],row['wt_sum_response_b'],row['counter_response_a'],row['counter_response_b']
    
    response_a_language_frequency={}
    response_b_language_frequency={}
    response_a_weighted_sum=0
    response_b_weighted_sum=0
      
    set_weight = row['prompt_weight']
    response_a = clean_before_wt_sum(row['response_a'])
    response_b = clean_before_wt_sum(row['response_b'])
    
    try:
        response_a_detector = Detector(response_a, quiet=True)
        response_b_detector = Detector(response_b, quiet=True)
        
        for language in response_a_detector.languages:
            if language.read_bytes==0 or language.name=='un':
                proportion=0
            else:
                proportion = language.read_bytes / sum(lang.read_bytes for lang in response_a_detector.languages)
            if (language.name not in response_a_language_frequency) and language.name!='un':
                response_a_language_frequency[language.name]=proportion
    
        for language in response_b_detector.languages:
            if language.read_bytes==0 or language.name=='un':
                proportion=0
            else:
                proportion = language.read_bytes / sum(lang.read_bytes for lang in response_b_detector.languages)
            if (language.name not in response_b_language_frequency) and language.name!='un':
                response_b_language_frequency[language.name]=proportion
    
        for key in list(response_a_language_frequency.keys()):
            if key in list(set_weight.keys()):
                response_a_weighted_sum = response_a_weighted_sum + set_weight[key]*response_a_language_frequency[key]
            else:
                response_a_weighted_sum = response_a_weighted_sum + 0*response_a_language_frequency[key]
    
        for key in list(response_b_language_frequency.keys()):
            if key in list(set_weight.keys()):
                response_b_weighted_sum = response_b_weighted_sum + set_weight[key]*response_b_language_frequency[key]
            else:
                response_b_weighted_sum = response_b_weighted_sum + 0*response_b_language_frequency[key]

    except:
        print(row['id'])
        # print(response_b_detector)

    return response_a_weighted_sum,response_b_weighted_sum,response_a_language_frequency,response_b_language_frequency

In [23]:
tqdm.pandas()
x_wt_list = list(df.progress_apply(club_detection_for_code, axis=1))

  2%|▏         | 882/48439 [00:00<00:15, 3140.55it/s]

0146c7de6e50ec02d9ccf81a68fb9f576cb128d8e259769fd40fc11be72fd842


 15%|█▌        | 7332/48439 [00:02<00:10, 4006.99it/s]

22e9d4c45fc6bd66dd49ebbf7a21ba43f47402f3e15d7e046fed051c80d5b1ed
2341d92bf76ce2d720ff0708a5a45567785f64262054c3cca7fc496dd959a6d5


 30%|███       | 14686/48439 [00:04<00:09, 3727.23it/s]

...

 89%|████████▉ | 43056/48439 [00:11<00:01, 3670.99it/s]

e0b65f52bc05f229a9dbb8f07820c0e251b1b0fd2c77492cfce969f62ea34453


 94%|█████████▍| 45684/48439 [00:12<00:00, 3701.72it/s]

ee53cda861b9e22a7e1d06473bc9f334bd98de111fd3a0cf96cba0181d58b897
ef538231819cb5c37f90f4a14e392c0be4f0f427f9154e813c1f4f3a260e9c47


100%|██████████| 48439/48439 [00:13<00:00, 3680.01it/s]


In [24]:
df['x_wt_meta'] = x_wt_list
df['wt_sum_response_a']= df['x_wt_meta'].apply(lambda x: x[0])
df['wt_sum_response_b']= df['x_wt_meta'].apply(lambda x: x[1])
df['counter_response_a']= df['x_wt_meta'].apply(lambda x: x[2])
df['counter_response_b']= df['x_wt_meta'].apply(lambda x: x[3])
del(df['x_wt_meta'])

In [6]:
# df.to_pickle("pixel_part_two")

### Similarity Score

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import AutoTokenizer, AutoModel
import torch

In [13]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

In [11]:
def embed_text(text):
    tokens = tokenizer(text, return_tensors='pt', padding=True, truncation=True)
    with torch.no_grad():
        embeddings = model(**tokens).pooler_output
    return embeddings


def deep_similarity(text1,text2,detected_language='english'):
    embeddings1 = embed_text(text1)
    embeddings2 = embed_text(text2)
    
    cosine_similarity = torch.nn.functional.cosine_similarity(embeddings1, embeddings2)
    return cosine_similarity.item()


def similarity_score(row):
    if row['type']=='logical_operations' or row['type']=='unknown':
        return -1,-1
    prompt = row['prompt']
    response_a = row['response_a']
    response_b = row['response_b']
    language = row['language']
    
    response_a_similarity_score = deep_similarity(prompt,response_a,language)
    response_b_similarity_score = deep_similarity(prompt,response_b,language)

    return response_a_similarity_score,response_b_similarity_score

In [ ]:
tqdm.pandas()
sm_list = list(df.progress_apply(similarity_score, axis=1))

 85%|████████▌ | 41180/48439 [1:29:33<17:25,  6.94it/s]  

In [4]:
!pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 12.5 MB/s eta 0:00:00


In [5]:
import logging
from sentence_transformers import SentenceTransformer, util

# Suppress unnecessary output
logging.getLogger("transformers").setLevel(logging.WARNING)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)

# Load a pre-trained multilingual model
model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(model_name)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
def compute_similarity(row):

    if row['type']=='logical_operations' or row['type']=='unknown':
        return -1,-1
        
    prompt = row['tr_prompt']
    response_a = row['tr_response_a']
    response_b = row['tr_response_b']

    prompt_embeddings = model.encode(prompt, convert_to_tensor=True)
    response_a_embeddings = model.encode(response_a, convert_to_tensor=True)
    response_b_embeddings = model.encode(response_b, convert_to_tensor=True)
    
    response_a_similarity_score = util.pytorch_cos_sim(prompt_embeddings, response_a_embeddings)
    response_b_similarity_score = util.pytorch_cos_sim(prompt_embeddings, response_b_embeddings)
    
    return response_a_similarity_score.item(), response_b_similarity_score.item()

In [9]:
tqdm.pandas()
sm_list = list(df.progress_apply(compute_similarity, axis=1))

100%|██████████| 48439/48439 [29:23<00:00, 27.46it/s]


In [13]:
df['sm_list'] = sm_list
df['similarity_score_response_a']= df['sm_list'].apply(lambda x: x[0])
df['similarity_score_response_b']= df['sm_list'].apply(lambda x: x[1])
del(df['sm_list'])

In [15]:
# df.to_pickle('pixel_part_three')

### Vocab Complexity

In [34]:
import re

def calculate_vocabulary(text):

    tokens = re.findall(r'\w+', text, re.UNICODE)
    normalized_tokens = [token.lower() for token in tokens]
    unique_words = list(set(normalized_tokens))
    return len(unique_words)


def vocab_complexity(row):
    if row['type']=='logical_operation' or row['type']=='unknown':
        return -1,-1,-1
        
    tr_prompt = row['tr_prompt']
    tr_response_a = row['tr_response_a']
    tr_response_b = row['tr_response_b']

    prompt_score=0
    response_a_score=0
    response_b_score=0
    
    if len(tr_prompt) != 0:
        prompt_score = round(calculate_vocabulary(tr_prompt)/len(tr_prompt),4)
    if len(tr_response_a)!=0:
        response_a_score = round(calculate_vocabulary(tr_response_a)/len(tr_response_a),4)
    if len(tr_response_b) != 0:
        response_b_score = round(calculate_vocabulary(tr_response_b)/len(tr_response_b),4)
    
    return prompt_score, response_a_score, response_b_score


In [39]:
tqdm.pandas()
vcs_list = list(df.progress_apply(vocab_complexity, axis=1))

100%|██████████| 48439/48439 [00:11<00:00, 4126.93it/s]


In [42]:
df['vcs_list'] = vcs_list
df['vocabulary_complexity_prompt']= df['vcs_list'].apply(lambda x: x[0])
df['vocabulary_complexity_response_a']= df['vcs_list'].apply(lambda x: x[1])
df['vocabulary_complexity_response_b']= df['vcs_list'].apply(lambda x: x[2])
del(df['vcs_list'])

### Code Comparision